# ExpresSo One True Compact WeightedEnsemble

This is the clean compact notebook for the final leakage-safe AutoGluon run, now with a switchable preprocessing variant for comparison.

The compact setup now uses only the first **5 known covariates** from the original feature list. Set `PREPROCESSING_VARIANT='manual_pipeline'` to explicitly apply `StandardScaler` and `OneHotEncoder` to those covariates before training, or set it to `'original'` to let AutoGluon handle the compact raw features.

The model family remains:

- `DirectTabular`
- `ChronosWithRegressor[bolt_small]`
- AutoGluon `WeightedEnsemble`

The goal is a readable, reproducible notebook that can run from the project root with only:

```text
super-ai-engineer-season-6-coffee-chain-hackathon/
```

No generated EDA/output files are required.


## Leakage Contract

The forecast rows contain three horizons. For each row, the model must only see target history available at its own decision date:

```text
1d decision date = forecast_date - 1 day
7d decision date = forecast_date - 7 days
1m decision date = forecast_date - 30 days
```

If a decision date falls after the training cutoff, it is capped at `2024-10-31`. During final prediction, this notebook predicts in batches by `effective_decision_date`, so a `1m` forecast for early November cannot use late-October target history that would not have existed yet.

Allowed future information:

- Official calendar/schedule lookup tables from the hackathon folder: `DATE_DIM`, `PROMOTION`, `LOCAL_EVENT`.
- Store/product metadata from the hackathon folder: `STORE`, `PRODUCT`.
- A fixed Bangkok monthly climatology proxy embedded in this notebook. This is not actual future observed weather.

Forbidden information:

- `test/ORDER.csv`, `test/TRANSACTION.csv`, `test/INVENTORY.csv`.
- Actual future observed weather.
- Future oil prices.
- Target-encoded local event lift tables built using validation/future targets.

## Feature Decision

We keep only the first 5 known covariates from the earlier feature set so the comparison stays small and easy to interpret.

Known covariates, available for both history and forecast horizon:

1. `day_of_week`
2. `is_holiday`
3. `is_payday`
4. `is_school_break`
5. `is_rainy_season`

Static features are kept compact and unchanged:

1. `store_id`
2. `category`
3. `horizon`
4. `horizon_days`
5. `neighborhood_type`

In `manual_pipeline`, the numeric flag covariates are scaled and `day_of_week` is one-hot encoded using transformers fitted only on the training period. We do not one-hot encode static features here, so the feature set stays small.


## References For External Constants

Runtime data dependencies are only local official hackathon CSV files. The only non-hackathon numeric constants are embedded in code and documented here.

| Item | Runtime source | Reference / reason | Leakage note |
|---|---|---|---|
| Official target history | `train/ORDER.csv`, `train/TRANSACTION.csv`, `test/PRODUCT.csv` | Hackathon dataset folder | Training history only, capped by decision date during prediction. |
| Calendar flags | `test/DATE_DIM.csv` | Hackathon dataset folder | Treated as known calendar/schedule information. |
| Promotions | `test/PROMOTION.csv` | Hackathon dataset folder | Treated as known campaign schedule. |
| Local events | `test/LOCAL_EVENT.csv` | Hackathon dataset folder | Treated as known event schedule; no target encoding. |
| Store/product static metadata | `test/STORE.csv`, `test/PRODUCT.csv` | Hackathon dataset folder | Static metadata, not future target. |
| Bangkok monthly climatology | Embedded constants in this notebook | ICAO Bangkok climatological table: [Climatological Information for Bangkok](https://www.icao.int/climatological-information-bangkok). The source reports monthly mean daily min/max temperature and precipitation. | Monthly climate normals only, not actual future weather for Nov-Dec 2024. |
| Serve-type temperature response | Embedded coefficients from train-only EDA hypothesis: hot `-0.685`, iced `+0.664`, blended `+0.666`, piece `+0.025`. | Used only as fixed product/category prior multiplied by climatology. | Does not read future sales; piece/hot/iced/blended come from `PRODUCT.csv`. |

## 0. Imports And Environment

We print the environment so every run is auditable. Use the `autogluon-ts` conda environment locally.

In [ ]:
# Colab-only install, if needed:
!pip -q install autogluon.timeseries "torch<2.10" torchvision torchaudio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.7/124.7 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 119.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7

In [ ]:
import importlib.metadata as importlib_metadata
import os
import platform
import re
import sys
import time
from pathlib import Path

os.environ.setdefault('LOKY_MAX_CPU_COUNT', '4')
os.environ.setdefault('MPLCONFIGDIR', '/private/tmp/matplotlib-expresso')

import numpy as np
import pandas as pd
from IPython.display import display
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)


def package_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return 'not installed'

ENVIRONMENT_REPORT = {
    'python_version': sys.version,
    'python_executable': sys.executable,
    'platform': platform.platform(),
    'machine': platform.machine(),
    'processor': platform.processor(),
    'cpu_count': os.cpu_count(),
    'working_directory': str(Path.cwd()),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'autogluon.timeseries': package_version('autogluon.timeseries'),
    'torch': package_version('torch'),
}
for key, value in ENVIRONMENT_REPORT.items():
    print(f'{key}: {value}')

python_version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
python_executable: /usr/bin/python3
platform: Linux-6.6.122+-x86_64-with-glibc2.35
machine: x86_64
processor: x86_64
cpu_count: 2
working_directory: /content
numpy: 2.0.2
pandas: 2.2.2
autogluon.timeseries: 1.5.0
torch: 2.9.1


## 1. Configuration

The output folder is separate from previous experiments. `TIME_LIMIT_SECONDS=900` is enough for this compact model on the local CPU run; increase it if you want more breathing room.

In [ ]:
import gdown
import zipfile
import os
from pathlib import Path

# URL and local path
url = 'https://drive.google.com/uc?id=1qJiohAo_VRw28_yDkCFtp21ltQjd7V0g'
output = 'super-ai-engineer-season-6-coffee-chain-hackathon.zip'
local_data_dir = Path('/content/super-ai-engineer-season-6-coffee-chain-hackathon')

# Download from Google Drive
if not os.path.exists(output):
    print("Downloading dataset from Google Drive...")
    gdown.download(url, output, quiet=False)

# Extract the zip file
if not local_data_dir.exists():
    print(f"Extracting {output} to {local_data_dir}...")
    with zipfile.ZipFile(output, 'r') as zip_ref:
        zip_ref.extractall(local_data_dir)
    print("Extraction complete.")
else:
    print(f"Directory {local_data_dir} already exists.")

Downloading...
From (original): https://drive.google.com/uc?id=1qJiohAo_VRw28_yDkCFtp21ltQjd7V0g
From (redirected): https://drive.google.com/uc?id=1qJiohAo_VRw28_yDkCFtp21ltQjd7V0g&confirm=t&uuid=2c00a2ad-2f5c-4238-8a0d-f03bcb974f33
To: /content/super-ai-engineer-season-6-coffee-chain-hackathon.zip
100%|██████████| 38.4M/38.4M [00:00<00:00, 53.9MB/s]


Extracting super-ai-engineer-season-6-coffee-chain-hackathon.zip to /content/super-ai-engineer-season-6-coffee-chain-hackathon...
Extraction complete.


Now that the data is downloaded and linked, the following configuration cell will successfully find the required CSV files.

In [ ]:
ROOT = Path.cwd()
DATA_DIR = ROOT / 'super-ai-engineer-season-6-coffee-chain-hackathon'
TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR = DATA_DIR / 'test'
OUTPUT_DIR = ROOT / 'outputs' / 'one_true_compact_weighted_ensemble'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FORECAST_START = pd.Timestamp('2024-11-01')
FORECAST_END = pd.Timestamp('2024-12-31')
TRAIN_END = pd.Timestamp('2024-10-31')
PREDICTION_LENGTH = (FORECAST_END - FORECAST_START).days + 1
HORIZON_TO_DAYS = {'1d': 1, '7d': 7, '1m': 30}

CAT_ORDER = [
    'Coffee',
    'Tea',
    'Bakery',
    'Savory Bakery',
    'Chocolate & Milk',
    'Juice & Smoothie',
    'Merchandise',
]
DOW_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

RUN_TRAINING = True
RUN_FEATURE_IMPORTANCE = True
TIME_LIMIT_SECONDS = 900
FEATURE_IMPORTANCE_TIME_LIMIT_SECONDS = 300

# Choose 'original' to reproduce the previous notebook, or 'manual_pipeline'
# to use explicit StandardScaler + OneHotEncoder preprocessing before AutoGluon.
PREPROCESSING_VARIANT = 'manual_pipeline'
VALID_PREPROCESSING_VARIANTS = {'original', 'manual_pipeline'}
if PREPROCESSING_VARIANT not in VALID_PREPROCESSING_VARIANTS:
    raise ValueError(f'PREPROCESSING_VARIANT must be one of {VALID_PREPROCESSING_VARIANTS}')

CHRONOS_BOLT_SMALL = {
    'ag_args': {'name_suffix': 'WithRegressor'},
    'model_path': 'bolt_small',
    'target_scaler': 'standard',
    'covariate_regressor': {'model_name': 'CAT', 'model_hyperparameters': {'iterations': 1000}},
}
HYPERPARAMETERS = {
    'DirectTabular': {},
    'Chronos': CHRONOS_BOLT_SMALL,
}
MODEL_PATH = OUTPUT_DIR / f'model_compact_weighted_ensemble_{PREPROCESSING_VARIANT}'

required_files = [
    DATA_DIR / 'sample_submission_with_id.csv',
    TRAIN_DIR / 'ORDER.csv',
    TRAIN_DIR / 'TRANSACTION.csv',
    TEST_DIR / 'PRODUCT.csv', TEST_DIR / 'STORE.csv', TEST_DIR / 'DATE_DIM.csv',
    TEST_DIR / 'PROMOTION.csv', TEST_DIR / 'LOCAL_EVENT.csv',
]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required official hackathon files: ' + ', '.join(missing))

print(f'Data dir: {DATA_DIR}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Prediction length: {PREDICTION_LENGTH} days')
print(f'Preprocessing variant: {PREPROCESSING_VARIANT}')

Data dir: /content/super-ai-engineer-season-6-coffee-chain-hackathon
Output dir: /content/outputs/one_true_compact_weighted_ensemble
Prediction length: 61 days


## 2. Load Official Hackathon Tables

We load only train target/history tables and test lookup/schedule tables. There is no read from test target/history files.

In [ ]:
order = pd.read_csv(TRAIN_DIR / 'ORDER.csv', parse_dates=['date'])
transaction = pd.read_csv(TRAIN_DIR / 'TRANSACTION.csv')
product = pd.read_csv(TEST_DIR / 'PRODUCT.csv')
store = pd.read_csv(TEST_DIR / 'STORE.csv', parse_dates=['opened_date'])
date_dim = pd.read_csv(TEST_DIR / 'DATE_DIM.csv', parse_dates=['date'])
promotion = pd.read_csv(TEST_DIR / 'PROMOTION.csv', parse_dates=['start_date', 'end_date'])
local_event = pd.read_csv(TEST_DIR / 'LOCAL_EVENT.csv', parse_dates=['date'])
sample = pd.read_csv(DATA_DIR / 'sample_submission_with_id.csv')

print({
    'order': order.shape,
    'transaction': transaction.shape,
    'product': product.shape,
    'store': store.shape,
    'date_dim': date_dim.shape,
    'promotion': promotion.shape,
    'local_event': local_event.shape,
    'sample': sample.shape,
})

{'order': (1376133, 7), 'transaction': (2858050, 6), 'product': (60, 7), 'store': (20, 8), 'date_dim': (731, 12), 'promotion': (31314, 10), 'local_event': (1440, 5), 'sample': (25620, 2)}


## 3. Target Panel

The competition target grain is daily `store_id x category`. We aggregate transaction line items through product category and fill missing active days with zero sales. Days before a store opened are excluded from training history.

In [ ]:
def slug_category(category):
    return pd.Series(category).astype(str).str.replace(r'[^A-Za-z0-9]+', '-', regex=True).str.strip('-')


def horizon_item_id(store_id, category, horizon):
    return pd.Series(store_id).astype(str) + '_' + slug_category(category) + '_' + pd.Series(horizon).astype(str)


def build_target_panel(order_df, transaction_df, product_df, store_df):
    txn = transaction_df.merge(
        order_df[['order_id', 'store_id', 'date']],
        on='order_id',
        how='left',
    ).merge(
        product_df[['product_id', 'category']],
        on='product_id',
        how='left',
    )
    raw = (
        txn.groupby(['store_id', 'category', 'date'], observed=True)['units_sold']
        .sum()
        .rename('units_sold')
        .reset_index()
    )
    all_dates = pd.date_range('2023-01-01', TRAIN_END, freq='D')
    full_index = pd.MultiIndex.from_product(
        [sorted(store_df['store_id'].unique()), CAT_ORDER, all_dates],
        names=['store_id', 'category', 'date'],
    )
    panel = raw.set_index(['store_id', 'category', 'date']).reindex(full_index, fill_value=0).reset_index()
    panel = panel.merge(store_df[['store_id', 'opened_date']], on='store_id', how='left')
    panel['effective_opened_date'] = panel['opened_date'].clip(lower=pd.Timestamp('2023-01-01'))
    panel = panel[panel['date'].ge(panel['effective_opened_date'])].copy()
    panel = panel.rename(columns={'date': 'timestamp'})
    return panel[['store_id', 'category', 'timestamp', 'units_sold']]


target_panel = build_target_panel(order, transaction, product, store)
print(target_panel.shape)
display(target_panel.head())

(92694, 4)


,store_id,category,timestamp,units_sold
0,1,Coffee,2023-01-01,100
1,1,Coffee,2023-01-02,140
2,1,Coffee,2023-01-03,109
3,1,Coffee,2023-01-04,116
4,1,Coffee,2023-01-05,136


## 4. Parse Submission Horizons

We parse each sample id into `store_id`, `category`, `forecast_date`, and `horizon`, then compute the decision date. This is the key to horizon-safe prediction later.

In [ ]:
def parse_sample_submission(sample_df):
    parsed = sample_df['id'].str.extract(
        r'^(?P<store_id>\d+)_(?P<category>.+)_(?P<forecast_date>\d{4}-\d{2}-\d{2})_(?P<horizon>1d|7d|1m)$'
    )
    if parsed.isna().any().any():
        bad_ids = sample_df.loc[parsed.isna().any(axis=1), 'id'].head(10).tolist()
        raise ValueError(f'Could not parse sample IDs: {bad_ids}')
    parsed['store_id'] = parsed['store_id'].astype(int)
    parsed['forecast_date'] = pd.to_datetime(parsed['forecast_date'])
    parsed['horizon_days'] = parsed['horizon'].map(HORIZON_TO_DAYS).astype(int)
    parsed['decision_date'] = parsed['forecast_date'] - pd.to_timedelta(parsed['horizon_days'], unit='D')
    parsed['effective_decision_date'] = parsed['decision_date'].where(parsed['decision_date'].le(TRAIN_END), TRAIN_END)
    parsed['item_id'] = horizon_item_id(parsed['store_id'], parsed['category'], parsed['horizon'])
    return pd.concat([sample_df[['id']], parsed], axis=1)


sample_parsed = parse_sample_submission(sample)
display(sample_parsed.head())
print(sample_parsed[['horizon', 'decision_date', 'effective_decision_date']].agg(['min', 'max']))

,id,store_id,category,forecast_date,horizon,horizon_days,decision_date,effective_decision_date,item_id
0,1_Bakery_2024-11-01_1d,1,Bakery,2024-11-01,1d,1,2024-10-31,2024-10-31,1_Bakery_1d
1,1_Bakery_2024-11-01_1m,1,Bakery,2024-11-01,1m,30,2024-10-02,2024-10-02,1_Bakery_1m
2,1_Bakery_2024-11-01_7d,1,Bakery,2024-11-01,7d,7,2024-10-25,2024-10-25,1_Bakery_7d
3,1_Bakery_2024-11-02_1d,1,Bakery,2024-11-02,1d,1,2024-11-01,2024-10-31,1_Bakery_1d
4,1_Bakery_2024-11-02_1m,1,Bakery,2024-11-02,1m,30,2024-10-03,2024-10-03,1_Bakery_1m


    horizon decision_date effective_decision_date
min      1d    2024-10-02              2024-10-02
max      7d    2024-12-30              2024-10-31


## 5. Compact Feature Builders

Each builder maps to one of the 14 selected known covariates. We intentionally do not build broad unused feature families.

In [ ]:
def add_calendar_features(df, date_dim_df):
    out = df.merge(date_dim_df.rename(columns={'date': 'timestamp'}), on='timestamp', how='left')
    out['day_of_week'] = pd.Categorical(out['day_of_week'], DOW_ORDER, ordered=True)
    out['dow_num'] = out['timestamp'].dt.dayofweek
    out['is_holiday'] = out['is_holiday'].fillna(False).astype(int)
    out['is_payday'] = out['is_payday'].fillna(False).astype(int)
    out['is_school_break'] = out['is_school_break'].fillna(False).astype(int)
    out['is_rainy_season'] = out['is_rainy_season'].fillna(False).astype(int)
    out['sin_doy'] = np.sin(2 * np.pi * out['timestamp'].dt.dayofyear / 365.25)
    out['cos_doy'] = np.cos(2 * np.pi * out['timestamp'].dt.dayofyear / 365.25)
    return out


def expand_promotions(promo_df, product_df):
    promo_prod = promo_df.merge(product_df[['product_id', 'category']], on='product_id', how='left')
    parts = []
    for row in promo_prod.itertuples(index=False):
        days = pd.date_range(row.start_date, row.end_date, freq='D')
        if len(days) == 0:
            continue
        parts.append(pd.DataFrame({
            'store_id': row.store_id,
            'category': row.category,
            'timestamp': days,
            'product_id': row.product_id,
            'discount_pct': row.discount_pct,
            'promo_type': row.promo_type,
        }))
    if not parts:
        return pd.DataFrame(columns=['store_id', 'category', 'timestamp'])
    daily = pd.concat(parts, ignore_index=True)
    return (
        daily.groupby(['store_id', 'category', 'timestamp'], observed=True)
        .agg(
            active_promo_products=('product_id', 'nunique'),
            max_discount_pct=('discount_pct', 'max'),
            promo_type_count=('promo_type', 'nunique'),
        )
        .reset_index()
    )


EVENT_TYPE_STORE_RELEVANCE = {
    'concert': {'mall': 1.0, 'tourist': 0.9, 'transit': 0.8, 'urban_residential': 0.7, 'university': 0.7, 'office': 0.4, 'hospital': 0.3, 'gas_station': 0.3},
    'music_festival': {'mall': 1.0, 'tourist': 1.0, 'transit': 0.8, 'urban_residential': 0.7, 'university': 0.7, 'office': 0.4, 'hospital': 0.3, 'gas_station': 0.3},
    'food_festival': {'mall': 0.9, 'tourist': 1.0, 'urban_residential': 0.8, 'transit': 0.7, 'university': 0.6, 'office': 0.5, 'hospital': 0.4, 'gas_station': 0.4},
    'market': {'urban_residential': 1.0, 'tourist': 0.8, 'transit': 0.7, 'university': 0.7, 'mall': 0.5, 'office': 0.4, 'hospital': 0.4, 'gas_station': 0.4},
    'book_fair': {'university': 1.0, 'mall': 0.8, 'tourist': 0.6, 'urban_residential': 0.6, 'office': 0.5, 'transit': 0.5, 'hospital': 0.3, 'gas_station': 0.2},
    'convention': {'office': 1.0, 'mall': 0.9, 'transit': 0.8, 'tourist': 0.7, 'university': 0.7, 'urban_residential': 0.5, 'hospital': 0.4, 'gas_station': 0.3},
    'sports': {'university': 0.9, 'urban_residential': 0.8, 'transit': 0.7, 'tourist': 0.6, 'mall': 0.5, 'gas_station': 0.5, 'office': 0.3, 'hospital': 0.3},
    'cultural': {'tourist': 1.0, 'urban_residential': 0.8, 'mall': 0.7, 'university': 0.6, 'transit': 0.6, 'gas_station': 0.4, 'hospital': 0.3, 'office': 0.3},
}
EVENT_TYPE_BASE_INTENSITY = {
    'music_festival': 1.20, 'concert': 1.10, 'food_festival': 1.05, 'convention': 0.95,
    'market': 0.85, 'book_fair': 0.80, 'sports': 0.75, 'cultural': 0.70,
}


def normalize_event_type(value):
    return str(value).strip().lower().replace(' ', '_')


def build_local_event_features(local_event_df, store_df):
    events = local_event_df.rename(columns={'date': 'timestamp'}).copy()
    events['timestamp'] = pd.to_datetime(events['timestamp'])
    events['event_type_norm'] = events['event_type'].map(normalize_event_type)
    events = events.merge(store_df[['store_id', 'neighborhood_type']], on='store_id', how='left')
    events['kaggle_event_intensity'] = events['event_type_norm'].map(EVENT_TYPE_BASE_INTENSITY).fillna(0.5)
    events['kaggle_event_store_relevance'] = [
        EVENT_TYPE_STORE_RELEVANCE.get(event_type, {}).get(neighborhood, 0.35)
        for event_type, neighborhood in zip(events['event_type_norm'], events['neighborhood_type'])
    ]
    return (
        events.groupby(['store_id', 'timestamp'], observed=True)
        .agg(
            kaggle_event_intensity_sum=('kaggle_event_intensity', 'sum'),
            kaggle_event_store_relevance_max=('kaggle_event_store_relevance', 'max'),
        )
        .reset_index()
    )


def build_bangkok_monthly_climatology(start, end):
    # Source: ICAO Bangkok climatological table. We use (mean min + mean max) / 2 as a monthly mean proxy.
    monthly = pd.DataFrame({
        'month': list(range(1, 13)),
        'mean_daily_min_c': [21.0, 23.3, 24.9, 26.1, 25.6, 25.4, 25.0, 24.9, 24.6, 24.3, 23.1, 20.8],
        'mean_daily_max_c': [32.0, 32.7, 33.7, 34.9, 34.0, 33.1, 32.7, 32.5, 32.3, 32.0, 31.6, 31.3],
    })
    monthly['temperature_2m_mean'] = (monthly['mean_daily_min_c'] + monthly['mean_daily_max_c']) / 2.0
    out = pd.DataFrame({'timestamp': pd.date_range(start, end, freq='D')})
    out['month'] = out['timestamp'].dt.month
    out = out.merge(monthly[['month', 'temperature_2m_mean']], on='month', how='left').drop(columns='month')
    return out


SERVE_TYPE_TEMP_RESPONSE = {
    'ร้อน': -0.685,
    'เย็น': 0.664,
    'ปั่น': 0.666,
    'ชิ้น': 0.025,
}


def build_serve_type_response(product_df):
    prod = product_df.copy()
    prod['serve_type_temp_response'] = prod['serve_type'].map(SERVE_TYPE_TEMP_RESPONSE).fillna(0.0)
    return (
        prod.groupby('category', observed=True)['serve_type_temp_response']
        .mean()
        .rename('serve_type_temp_response_score')
        .reset_index()
    )


promo_features = expand_promotions(promotion, product)
local_event_features = build_local_event_features(local_event, store)
weather_features = build_bangkok_monthly_climatology(pd.Timestamp('2023-01-01'), FORECAST_END)
serve_type_features = build_serve_type_response(product)

print('promo_features', promo_features.shape)
print('local_event_features', local_event_features.shape)
print('weather_features', weather_features.shape)
print('serve_type_features', serve_type_features.shape)

promo_features (39901, 6)
local_event_features (1371, 4)
weather_features (731, 2)
serve_type_features (7, 2)


## 6. Build The AutoGluon Frame

We build an AutoGluon item for each `store_id x category x horizon`. The same target history is repeated across horizons, but prediction context is cut separately by horizon decision date.

In [ ]:
def make_full_item_calendar(store_df, categories, horizons, start, end):
    dates = pd.date_range(start, end, freq='D')
    full = pd.MultiIndex.from_product(
        [sorted(store_df['store_id'].unique()), categories, list(horizons.keys()), dates],
        names=['store_id', 'category', 'horizon', 'timestamp'],
    ).to_frame(index=False)
    full['horizon_days'] = full['horizon'].map(horizons).astype(int)
    full['decision_date'] = full['timestamp'] - pd.to_timedelta(full['horizon_days'], unit='D')
    full['effective_decision_date'] = full['decision_date'].where(full['decision_date'].le(TRAIN_END), TRAIN_END)
    full['item_id'] = horizon_item_id(full['store_id'], full['category'], full['horizon'])
    return full


def build_model_frame():
    full = make_full_item_calendar(store, CAT_ORDER, HORIZON_TO_DAYS, pd.Timestamp('2023-01-01'), FORECAST_END)
    frame = full.merge(target_panel, on=['store_id', 'category', 'timestamp'], how='left')
    frame = frame.merge(store[['store_id', 'neighborhood_type']], on='store_id', how='left')
    frame = add_calendar_features(frame, date_dim)
    frame = frame.merge(promo_features, on=['store_id', 'category', 'timestamp'], how='left')
    frame = frame.merge(local_event_features, on=['store_id', 'timestamp'], how='left')
    frame = frame.merge(weather_features, on='timestamp', how='left')
    frame = frame.merge(serve_type_features, on='category', how='left')

    for col in ['active_promo_products', 'max_discount_pct', 'promo_type_count', 'kaggle_event_store_relevance_max', 'kaggle_event_intensity_sum']:
        frame[col] = pd.to_numeric(frame[col], errors='coerce').fillna(0)
    frame['serve_type_temp_response_score'] = pd.to_numeric(frame['serve_type_temp_response_score'], errors='coerce').fillna(0)
    frame['serve_type_temp_response_x_temperature_mean'] = frame['serve_type_temp_response_score'] * frame['temperature_2m_mean']

    frame['category'] = pd.Categorical(frame['category'], CAT_ORDER, ordered=True)
    frame['horizon'] = pd.Categorical(frame['horizon'], list(HORIZON_TO_DAYS.keys()), ordered=True)
    frame['neighborhood_type'] = frame['neighborhood_type'].astype('category')
    return frame.copy()


model_frame = build_model_frame()
print(model_frame.shape)
display(model_frame.head())

(307020, 32)


,store_id,category,horizon,timestamp,horizon_days,decision_date,effective_decision_date,item_id,units_sold,neighborhood_type,day_of_week,week_number,month,quarter,year,is_weekend,is_holiday,holiday_name,is_school_break,is_payday,is_rainy_season,dow_num,sin_doy,cos_doy,active_promo_products,max_discount_pct,promo_type_count,kaggle_event_intensity_sum,kaggle_event_store_relevance_max,temperature_2m_mean,serve_type_temp_response_score,serve_type_temp_response_x_temperature_mean
0,1,Coffee,1d,2023-01-01,1,2022-12-31,2022-12-31,1_Coffee_1d,100.0,university,Sunday,52,1,1,2023,True,1,วันขึ้นปีใหม่,0,0,0,6,0.017202,0.999852,0.0,0.0,0.0,0.0,0.0,26.5,0.108882,2.885382
1,1,Coffee,1d,2023-01-02,1,2023-01-01,2023-01-01,1_Coffee_1d,140.0,university,Monday,1,1,1,2023,False,0,NaN,0,0,0,0,0.034398,0.999408,0.0,0.0,0.0,0.0,0.0,26.5,0.108882,2.885382
2,1,Coffee,1d,2023-01-03,1,2023-01-02,2023-01-02,1_Coffee_1d,109.0,university,Tuesday,1,1,1,2023,False,0,NaN,0,0,0,1,0.051584,0.998669,0.0,0.0,0.0,0.0,0.0,26.5,0.108882,2.885382
3,1,Coffee,1d,2023-01-04,1,2023-01-03,2023-01-03,1_Coffee_1d,116.0,university,Wednesday,1,1,1,2023,False,0,NaN,0,0,0,2,0.068755,0.997634,0.0,0.0,0.0,0.0,0.0,26.5,0.108882,2.885382
4,1,Coffee,1d,2023-01-05,1,2023-01-04,2023-01-04,1_Coffee_1d,136.0,university,Thursday,1,1,1,2023,False,0,NaN,0,0,0,3,0.085906,0.996303,0.0,0.0,0.0,0.7,0.6,26.5,0.108882,2.885382


## 7. Select 5 Features And Optional Manual Preprocessing Pipeline

Set `PREPROCESSING_VARIANT='original'` for the compact raw 5-feature AutoGluon path, or `manual_pipeline` to scale the numeric flags and one-hot encode `day_of_week` before fitting. Static features stay as the same 5 compact fields in both variants.


In [ ]:
base_known_covariates = [
    'day_of_week',
    'is_holiday',
    'is_payday',
    'is_school_break',
    'is_rainy_season',
]
base_static_feature_cols = ['store_id', 'category', 'horizon', 'horizon_days', 'neighborhood_type']

manual_numeric_known_covariates = [
    'is_holiday',
    'is_payday',
    'is_school_break',
    'is_rainy_season',
]
manual_categorical_known_covariates = ['day_of_week']

missing_known = sorted(set(base_known_covariates) - set(model_frame.columns))
missing_static = sorted(set(base_static_feature_cols) - set(model_frame.columns))
if missing_known or missing_static:
    raise AssertionError({'missing_known': missing_known, 'missing_static': missing_static})


def make_one_hot_encoder():
    # sklearn renamed sparse -> sparse_output in newer versions; this keeps Colab/local runs compatible.
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


def prefixed_feature_names(prefix, encoder, input_cols):
    raw_names = encoder.get_feature_names_out(input_cols)
    clean_names = [re.sub(r'[^A-Za-z0-9_]+', '_', str(name)).strip('_') for name in raw_names]
    return [f'{prefix}{name}' for name in clean_names]


def build_static_features(frame):
    static = frame.groupby('item_id', observed=True)[base_static_feature_cols].first()
    for col in ['store_id', 'category', 'horizon', 'neighborhood_type']:
        static[col] = static[col].astype(str).astype('category')
    return static


def build_manual_preprocessed_frame(frame):
    train_mask = frame['timestamp'].le(TRAIN_END) & frame['units_sold'].notna()
    train_part = frame.loc[train_mask].copy()
    out = frame.copy()

    known_scaler = StandardScaler()
    known_scaled_cols = [f'scaled_{col}' for col in manual_numeric_known_covariates]
    known_scaler.fit(train_part[manual_numeric_known_covariates])
    out[known_scaled_cols] = known_scaler.transform(out[manual_numeric_known_covariates])

    dow_encoder = make_one_hot_encoder()
    dow_encoder.fit(train_part[manual_categorical_known_covariates].astype(str))
    dow_cols = prefixed_feature_names('oh_', dow_encoder, manual_categorical_known_covariates)
    out[dow_cols] = dow_encoder.transform(out[manual_categorical_known_covariates].astype(str))

    processed_known_covariates = known_scaled_cols + dow_cols
    processed_static_features = build_static_features(out)
    fitted_pipeline = {
        'known_scaler': known_scaler,
        'dow_encoder': dow_encoder,
        'known_scaled_cols': known_scaled_cols,
        'dow_cols': dow_cols,
    }
    return out, processed_known_covariates, processed_static_features, fitted_pipeline


if PREPROCESSING_VARIANT == 'manual_pipeline':
    model_frame_for_training, known_covariates, static_features, preprocessing_pipeline = build_manual_preprocessed_frame(model_frame)
    preprocessing_description = 'Compact manual preprocessing: StandardScaler for 4 calendar flags + OneHotEncoder for day_of_week, fitted on train period only. Static features remain compact.'
else:
    model_frame_for_training = model_frame.copy()
    known_covariates = base_known_covariates.copy()
    static_features = build_static_features(model_frame_for_training)
    preprocessing_pipeline = None
    preprocessing_description = 'Original compact preprocessing: 5 raw known covariates; AutoGluon handles categorical/static processing internally.'

train_df = model_frame_for_training[
    model_frame_for_training['timestamp'].le(TRAIN_END) & model_frame_for_training['units_sold'].notna()
].copy()
future_cov_df = model_frame_for_training[
    model_frame_for_training['timestamp'].between(FORECAST_START, FORECAST_END)
].copy()

train_cols = ['item_id', 'timestamp', 'units_sold'] + known_covariates
future_cols = ['item_id', 'timestamp'] + known_covariates

train_tsdf = TimeSeriesDataFrame.from_data_frame(
    train_df[train_cols],
    id_column='item_id',
    timestamp_column='timestamp',
)
train_tsdf.static_features = static_features

future_known_covariates = TimeSeriesDataFrame.from_data_frame(
    future_cov_df[future_cols],
    id_column='item_id',
    timestamp_column='timestamp',
)

print('Preprocessing:', PREPROCESSING_VARIANT)
print(preprocessing_description)
print('Known covariates:', len(known_covariates), known_covariates)
print('Static features:', len(static_features.columns), static_features.columns.tolist())
print('train_tsdf:', train_tsdf.shape)
print('future_known_covariates:', future_known_covariates.shape)
print('static_features:', static_features.shape)


Known covariates: 14 ['day_of_week', 'is_holiday', 'is_payday', 'is_school_break', 'is_rainy_season', 'sin_doy', 'cos_doy', 'active_promo_products', 'max_discount_pct', 'promo_type_count', 'kaggle_event_store_relevance_max', 'kaggle_event_intensity_sum', 'temperature_2m_mean', 'serve_type_temp_response_x_temperature_mean']
Static features: 5 ['store_id', 'category', 'horizon', 'horizon_days', 'neighborhood_type']
train_tsdf: (278082, 15)
future_known_covariates: (25620, 14)
static_features: (420, 5)


## 8. Pre-Training Leakage Audit

This is a guardrail before fitting. It checks that the compact feature set is exactly the intended one, that no oil/future-target-derived fields enter, and that forecast covariates cover the required horizon.

In [ ]:
def run_pretraining_leakage_audit():
    forbidden_known = [
        col for col in known_covariates
        if col.startswith(('ptt_', 'bangchak_'))
        or 'ptt_vs_bangchak' in col
        or 'stockout' in col
        or col in {
            'kaggle_event_type_category_lift_max',
            'kaggle_event_type_category_obs_max',
            'kaggle_event_type_store_category_lift_max',
            'kaggle_event_type_store_category_obs_max',
        }
    ]
    expected_known = 5 if PREPROCESSING_VARIANT == 'original' else len(known_covariates)
    expected_static = 5
    audit = {
        'known_covariate_count_matches_variant': len(known_covariates) == expected_known,
        'static_feature_count_matches_variant': len(static_features.columns) == expected_static,
        'manual_pipeline_has_scaled_features': PREPROCESSING_VARIANT != 'manual_pipeline' or any(col.startswith('scaled_') for col in known_covariates),
        'manual_pipeline_has_one_hot_features': PREPROCESSING_VARIANT != 'manual_pipeline' or any(col.startswith('oh_') for col in known_covariates),
        'no_forbidden_known_covariates': len(forbidden_known) == 0,
        'train_rows_stop_at_train_end': train_df['timestamp'].max() <= TRAIN_END,
        'future_covariates_cover_forecast_start': future_cov_df['timestamp'].min() <= FORECAST_START,
        'future_covariates_cover_forecast_end': future_cov_df['timestamp'].max() >= FORECAST_END,
        'sample_decisions_stop_at_train_end': sample_parsed['effective_decision_date'].max() <= TRAIN_END,
        'no_missing_future_known_covariates': not future_cov_df[known_covariates].isna().any().any(),
    }
    failed = [name for name, ok in audit.items() if not bool(ok)]
    if failed:
        raise AssertionError({'failed_leakage_checks': failed, 'forbidden_known_covariates': forbidden_known})
    audit_df = pd.DataFrame({'check': list(audit.keys()), 'passed': list(audit.values())})
    audit_df.to_csv(OUTPUT_DIR / f'pretraining_leakage_audit_{PREPROCESSING_VARIANT}.csv', index=False)
    print('Pre-training leakage audit: PASS')
    display(audit_df)


run_pretraining_leakage_audit()


Pre-training leakage audit: PASS


,check,passed
0,exactly_14_known_covariates,True
1,exactly_5_static_features,True
2,no_forbidden_known_covariates,True
3,train_rows_stop_at_train_end,True
4,future_covariates_cover_forecast_start,True
5,future_covariates_cover_forecast_end,True
6,sample_decisions_stop_at_train_end,True
7,no_missing_future_known_covariates,True


## 9. Fit WeightedEnsemble

We use `DirectTabular + ChronosWithRegressor[bolt_small]` because the timing comparison showed:

- DirectTabular only: fastest, MAE around `7.95`.
- Chronos only: better, MAE around `7.79`.
- WeightedEnsemble: best compact result, MAE around `7.73`, with weights roughly Chronos-heavy plus a DirectTabular correction.

This model uses the compact 14/5 feature set, not the larger earlier feature universe.

In [ ]:
if RUN_TRAINING:
    fit_start = time.perf_counter()
    predictor = TimeSeriesPredictor(
        target='units_sold',
        prediction_length=PREDICTION_LENGTH,
        freq='D',
        eval_metric='MAE',
        known_covariates_names=known_covariates,
        path=str(MODEL_PATH),
    ).fit(
        train_data=train_tsdf,
        time_limit=TIME_LIMIT_SECONDS,
        hyperparameters=HYPERPARAMETERS,
        enable_ensemble=True,
        num_val_windows=2,
        refit_every_n_windows=1,
    )
    fit_wall_seconds = time.perf_counter() - fit_start
    leaderboard = predictor.leaderboard()
    leaderboard.to_csv(OUTPUT_DIR / 'leaderboard.csv', index=False)
    print(f'Fit wall seconds: {fit_wall_seconds:.2f}')
    display(leaderboard)
else:
    predictor = TimeSeriesPredictor.load(str(MODEL_PATH))
    leaderboard = predictor.leaderboard()
    display(leaderboard)

Beginning AutoGluon training... Time limit = 900s
AutoGluon will save models to '/content/outputs/one_true_compact_weighted_ensemble/model_compact_weighted_ensemble'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       10.70 GB / 12.67 GB (84.5%)
Disk Space Avail:   65.00 GB / 112.64 GB (57.7%)

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': MAE,
 'freq': 'D',
 'hyperparameters': {'Chronos': {'ag_args': {'name_suffix': 'WithRegressor'},
                                 'covariate_regressor': {'model_hyperparameters': {'iterations': 1000},
                                       

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/191M [00:00<?, ?B/s]

	-7.9024       = Validation score (-MAE)
	138.61  s     = Training runtime
	1.46    s     = Validation (prediction) runtime
Fitting 1 ensemble(s), in 1 layers.
Training ensemble model WeightedEnsemble. Training for up to 719.5s.
	Ensemble weights: {'ChronosWithRegressor[bolt_small]': 0.59, 'DirectTabular': 0.41}
	-7.7955       = Validation score (-MAE)
	1.83    s     = Training runtime
	2.92    s     = Validation (prediction) runtime
Training complete. Models trained: ['DirectTabular', 'ChronosWithRegressor[bolt_small]', 'WeightedEnsemble']
Total runtime: 178.70 s
Best model: WeightedEnsemble
Best model score: -7.7955


Fit wall seconds: 182.67


,model,score_val,pred_time_val,fit_time_marginal,fit_order
0,WeightedEnsemble,-7.795528,2.920751,1.825575,3
1,ChronosWithRegressor[bolt_small],-7.902389,1.455787,138.606860,2
2,DirectTabular,-7.983465,1.458111,28.582436,1


## 10. Horizon-Safe Prediction And Submission

The predictor is called once per effective decision date. Each call receives only target history through that cutoff plus known future covariates for the next 61 days.

In [ ]:
def make_tsdf_for_cutoff(cutoff, item_ids=None):
    cutoff = pd.Timestamp(cutoff)
    history = train_df[train_df['timestamp'].le(cutoff)].copy()
    if item_ids is not None:
        history = history[history['item_id'].isin(item_ids)].copy()
    if history.empty:
        raise ValueError(f'No target history available through cutoff {cutoff.date()}')
    tsdf = TimeSeriesDataFrame.from_data_frame(
        history[train_cols],
        id_column='item_id',
        timestamp_column='timestamp',
    )
    tsdf.static_features = static_features.loc[static_features.index.intersection(history['item_id'].unique())]
    return tsdf


def make_future_covariates_for_cutoff(cutoff, item_ids=None):
    cutoff = pd.Timestamp(cutoff)
    horizon_end = cutoff + pd.Timedelta(days=PREDICTION_LENGTH)
    future = model_frame_for_training[model_frame_for_training['timestamp'].between(cutoff + pd.Timedelta(days=1), horizon_end)].copy()
    if item_ids is not None:
        future = future[future['item_id'].isin(item_ids)].copy()
    if future.empty:
        raise ValueError(f'No known covariates available after cutoff {cutoff.date()}')
    return TimeSeriesDataFrame.from_data_frame(
        future[future_cols],
        id_column='item_id',
        timestamp_column='timestamp',
    )


def predict_horizon_safe_submission(predictor, sample_df):
    pieces = []
    audit_rows = []
    batch_rows = []
    sample_work = sample_df.copy()
    sample_work['effective_decision_date'] = pd.to_datetime(sample_work['effective_decision_date'])
    sample_work['forecast_date'] = pd.to_datetime(sample_work['forecast_date'])

    for cutoff, group in sample_work.groupby('effective_decision_date', sort=True):
        cutoff = pd.Timestamp(cutoff)
        item_ids = sorted(group['item_id'].unique())
        max_requested_offset = int((group['forecast_date'].max() - cutoff).days)
        if max_requested_offset > PREDICTION_LENGTH:
            raise ValueError(f'Cutoff {cutoff.date()} needs {max_requested_offset} days but prediction_length={PREDICTION_LENGTH}')

        context_tsdf = make_tsdf_for_cutoff(cutoff, item_ids=item_ids)
        future_covariates = make_future_covariates_for_cutoff(cutoff, item_ids=item_ids)
        preds = predictor.predict(context_tsdf, known_covariates=future_covariates)
        pred_mean = (
            preds.reset_index()[['item_id', 'timestamp', 'mean']]
            .rename(columns={'timestamp': 'forecast_date', 'mean': 'units_sold_predicted'})
        )
        merged = group.merge(pred_mean, on=['item_id', 'forecast_date'], how='left')
        missing = int(merged['units_sold_predicted'].isna().sum())
        if missing:
            raise ValueError(f'Missing {missing} predictions for cutoff {cutoff.date()}')
        merged['units_sold_predicted'] = merged['units_sold_predicted'].clip(lower=0)
        pieces.append(merged)

        context_max = context_tsdf.reset_index()['timestamp'].max()
        future_max = future_covariates.reset_index()['timestamp'].max()
        audit_rows.append({
            'effective_decision_date': cutoff,
            'rows': len(group),
            'items': len(item_ids),
            'context_max_timestamp': context_max,
            'future_covariate_max_timestamp': future_max,
            'max_requested_offset': max_requested_offset,
            'history_after_effective_decision': context_max > cutoff,
            'dynamic_covariate_after_context': False,
            'forecast_outside_prediction_window': max_requested_offset > PREDICTION_LENGTH,
        })
        batch_rows.append({
            'effective_decision_date': cutoff,
            'rows': len(group),
            'items': len(item_ids),
            'context_max_timestamp': context_max,
            'max_requested_offset': max_requested_offset,
        })

    out = pd.concat(pieces, ignore_index=True).sort_values('id')
    audit = pd.DataFrame(audit_rows)
    batches = pd.DataFrame(batch_rows)

    leak_count = int(audit['history_after_effective_decision'].sum())
    outside_count = int(audit['forecast_outside_prediction_window'].sum())
    if leak_count or outside_count:
        raise AssertionError(f'Horizon audit failed: history leaks={leak_count}, outside window={outside_count}')
    return out, audit, batches


prediction_start = time.perf_counter()
strict_submission, horizon_audit, horizon_batches = predict_horizon_safe_submission(predictor, sample_parsed)
prediction_wall_seconds = time.perf_counter() - prediction_start

final_submission = strict_submission[['id', 'units_sold_predicted']].copy()
submission_path = OUTPUT_DIR / f'submission_compact_weighted_ensemble_{PREPROCESSING_VARIANT}.csv'
audit_path = OUTPUT_DIR / f'horizon_safe_prediction_audit_{PREPROCESSING_VARIANT}.csv'
batch_path = OUTPUT_DIR / f'horizon_safe_prediction_batches_{PREPROCESSING_VARIANT}.csv'

final_submission.to_csv(submission_path, index=False)
horizon_audit.to_csv(audit_path, index=False)
horizon_batches.to_csv(batch_path, index=False)

print(f'Prediction wall seconds: {prediction_wall_seconds:.2f}')
print(submission_path)
print(audit_path)
print(batch_path)
print('History-after-decision leaks:', int(horizon_audit['history_after_effective_decision'].sum()))
print('Prediction-window violations:', int(horizon_audit['forecast_outside_prediction_window'].sum()))
display(final_submission.head())

Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble
Model not specified in predict, will 

Prediction wall seconds: 66.84
/content/outputs/one_true_compact_weighted_ensemble/submission_compact_weighted_ensemble.csv
/content/outputs/one_true_compact_weighted_ensemble/horizon_safe_prediction_audit.csv
/content/outputs/one_true_compact_weighted_ensemble/horizon_safe_prediction_batches.csv
History-after-decision leaks: 0
Prediction-window violations: 0


,id,units_sold_predicted
14224,10_Bakery_2024-11-01_1d,58.963407
63,10_Bakery_2024-11-01_1m,56.322385
3346,10_Bakery_2024-11-01_7d,56.101716
14225,10_Bakery_2024-11-02_1d,81.541992
203,10_Bakery_2024-11-02_1m,79.315772


## 11. Sanity Checks

These are not leaderboard scores. They are quick checks that the forecast distribution is plausible versus recent training history and that horizon-specific rows can differ.

In [ ]:
recent_unique = train_df.drop_duplicates(['store_id', 'category', 'timestamp'])
recent = recent_unique[recent_unique['timestamp'].ge(TRAIN_END - pd.Timedelta(days=60))]
recent_summary = recent.groupby('category', observed=True)['units_sold'].agg(['mean', 'median', 'sum']).reset_index()
forecast_by_category = strict_submission.groupby('category', observed=True)['units_sold_predicted'].agg(['mean', 'median', 'sum']).reset_index()
check = recent_summary.merge(forecast_by_category, on='category', suffixes=('_recent_61d', '_forecast_rows'))
check['mean_ratio_forecast_to_recent'] = check['mean_forecast_rows'] / check['mean_recent_61d']
check.to_csv(OUTPUT_DIR / f'forecast_category_sanity_check_{PREPROCESSING_VARIANT}.csv', index=False)

audit_summary = pd.DataFrame([{
    'preprocessing_variant': PREPROCESSING_VARIANT,
    'known_covariate_count': len(known_covariates),
    'static_feature_count': len(static_features.columns),
    'fit_wall_seconds': globals().get('fit_wall_seconds', np.nan),
    'prediction_wall_seconds': prediction_wall_seconds,
    'best_model': leaderboard.sort_values('score_val', ascending=False).iloc[0]['model'],
    'validation_mae': -float(leaderboard.sort_values('score_val', ascending=False).iloc[0]['score_val']),
}])
audit_summary.to_csv(OUTPUT_DIR / f'run_summary_{PREPROCESSING_VARIANT}.csv', index=False)

display(audit_summary)
display(check)

,known_covariate_count,static_feature_count,fit_wall_seconds,prediction_wall_seconds,best_model,validation_mae
0,14,5,182.669142,66.838865,WeightedEnsemble,7.795528


,category,mean_recent_61d,median_recent_61d,sum_recent_61d,mean_forecast_rows,median_forecast_rows,sum_forecast_rows,mean_ratio_forecast_to_recent
0,Coffee,143.717213,129.5,175335.0,164.547260,149.308299,602242.973420,1.144938
1,Tea,44.838525,39.0,54703.0,51.575141,45.479074,188765.017688,1.150242
2,Bakery,35.202459,31.0,42947.0,40.280350,36.986995,147426.079468,1.144248
3,Savory Bakery,28.209016,25.0,34415.0,32.079953,29.523585,117412.626435,1.137223
4,Chocolate & Milk,20.626230,18.0,25164.0,23.572170,21.286357,86274.141104,1.142825
5,Juice & Smoothie,12.401639,10.5,15130.0,14.152798,12.535246,51799.241993,1.141204
6,Merchandise,7.340984,6.0,8956.0,8.272190,7.550626,30276.217116,1.126850


## 12. Optional Feature Importance

Off by default because this notebook is meant to be compact and fast. Turn `RUN_FEATURE_IMPORTANCE=True` if you need a compact-model permutation importance plot.

In [ ]:
if RUN_FEATURE_IMPORTANCE:
    feature_importance = predictor.feature_importance(
        data=train_tsdf,
        method='permutation',
        subsample_size=50,
        num_iterations=1,
        time_limit=FEATURE_IMPORTANCE_TIME_LIMIT_SECONDS,
    )
    feature_importance_sorted = feature_importance.sort_values('importance', ascending=False)
    feature_importance_path = OUTPUT_DIR / 'feature_importance.csv'
    feature_importance_sorted_path = OUTPUT_DIR / 'feature_importance_sorted.csv'
    feature_importance.to_csv(feature_importance_path)
    feature_importance_sorted.to_csv(feature_importance_sorted_path)
    print(feature_importance_path)
    print(feature_importance_sorted_path)
    display(feature_importance_sorted)
else:
    print('RUN_FEATURE_IMPORTANCE=False; skipped feature importance.')

RUN_FEATURE_IMPORTANCE=False; skipped feature importance.


## 13. Write Reference Manifest

The manifest is useful when handing the notebook to teammates. It records feature choices, data sources, and leakage decisions.

In [ ]:
reference_markdown = f"""# Compact 5-Feature WeightedEnsemble Feature Manifest ({PREPROCESSING_VARIANT})

Generated by `Expresso-One-True-Compact-WeightedEnsemble.ipynb`.

## Model

- AutoGluon `TimeSeriesPredictor`
- `prediction_length={PREDICTION_LENGTH}`
- Hyperparameters: `DirectTabular` + `ChronosWithRegressor[bolt_small]`
- Ensemble enabled: `WeightedEnsemble`

## Preprocessing

Variant: `{PREPROCESSING_VARIANT}`

{preprocessing_description}

## Known Covariates ({len(known_covariates)})

```text
{chr(10).join(known_covariates)}
```

## Static Features ({len(static_features.columns)})

```text
{chr(10).join(static_features.columns.astype(str))}
```

## Data Sources

| Source | Used for | Runtime file dependency | Leakage stance |
|---|---|---|---|
| Hackathon train `ORDER.csv` | Target history dates/store/order joins | `super-ai-engineer-season-6-coffee-chain-hackathon/train/ORDER.csv` | Training history only; context cut by horizon decision date. |
| Hackathon train `TRANSACTION.csv` | Units sold by product/order | `super-ai-engineer-season-6-coffee-chain-hackathon/train/TRANSACTION.csv` | Training history only; context cut by horizon decision date. |
| Hackathon test `PRODUCT.csv` | Category and serve type metadata | `super-ai-engineer-season-6-coffee-chain-hackathon/test/PRODUCT.csv` | Static lookup, not target. |
| Hackathon test `STORE.csv` | Store metadata and open dates | `super-ai-engineer-season-6-coffee-chain-hackathon/test/STORE.csv` | Static lookup, not target. |
| Hackathon test `DATE_DIM.csv` | Calendar flags | `super-ai-engineer-season-6-coffee-chain-hackathon/test/DATE_DIM.csv` | Known calendar/schedule data. |
| Hackathon test `PROMOTION.csv` | Promotion schedule features | `super-ai-engineer-season-6-coffee-chain-hackathon/test/PROMOTION.csv` | Known schedule data. |
| Hackathon test `LOCAL_EVENT.csv` | Local event schedule features | `super-ai-engineer-season-6-coffee-chain-hackathon/test/LOCAL_EVENT.csv` | Known schedule data; no target encoding. |
| ICAO Bangkok climatology | Monthly temperature proxy | Embedded constants; no runtime file/download | Climate normals only, not actual future observed weather. Reference: https://www.icao.int/climatological-information-bangkok |
| Serve-type temperature response | Category temperature interaction | Embedded train-EDA coefficients + `PRODUCT.csv` serve_type | Fixed prior; no future target read. |

## Explicit Exclusions

- No future actual weather files/API.
- No oil price features.
- No test target/history files.
- No target-derived local-event lift features.
- No stockout summary features in the compact model.
"""
ref_path = OUTPUT_DIR / f'compact_weighted_ensemble_feature_manifest_{PREPROCESSING_VARIANT}.md'
env_path = OUTPUT_DIR / f'environment_report_{PREPROCESSING_VARIANT}.csv'
pd.DataFrame([ENVIRONMENT_REPORT]).to_csv(env_path, index=False)
ref_path.write_text(reference_markdown, encoding='utf-8')
print(ref_path)
print(env_path)

/content/outputs/one_true_compact_weighted_ensemble/compact_weighted_ensemble_feature_manifest.md
/content/outputs/one_true_compact_weighted_ensemble/environment_report.csv
